# Modern-target prediction dataset

Design B from the temporal check: historical inspections (2001-2023) are only used to build feature history, and prediction targets are restricted to the current DineSafe period. Only current-period inspections are used as targets. Historical inspections can still contribute to the features because they would have been known before the target inspection.

In [1]:
import pandas as pd
import numpy as np

CUTOFF = pd.Timestamp("2023-11-10")  # first date in the current DineSafe CSV

df = pd.read_csv("../outputs/inspection_level_longitudinal.csv", parse_dates=["inspectionDate"], low_memory=False)
df = df.sort_values(["estId", "inspectionDate"]).reset_index(drop=True)
print("Loaded rows:", len(df))
df.head()

Loaded rows: 262547


,estId,oldEstId,estName,address,latitude,longitude,inspectionDate,inspectionStatus,n_infractions,n_minor,n_significant,n_crucial,source
0,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2025-05-22,Pass,0,0,0,0,current
1,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2025-10-02,Pass,2,2,0,0,current
2,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2026-02-03,Pass,1,0,1,0,current
3,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2026-07-14,Conditional Pass,2,0,2,0,current
4,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2026-07-21,Pass,0,0,0,0,current


## Base history features

Same as the previous prediction-event notebook: previous inspection outcome, infraction counts, days since last inspection, and how many inspections have been seen so far. Computed with `shift(1)` and `cumcount()` within each establishment's full sorted history (historical + current), so a current-period target can pick up features from historical rows.

In [2]:
grp = df.groupby("estId")

df["prev_inspectionDate"] = grp["inspectionDate"].shift(1)
df["prev_status"] = grp["inspectionStatus"].shift(1)
df["prev_total_infractions"] = grp["n_infractions"].shift(1)
df["prev_minor"] = grp["n_minor"].shift(1)
df["prev_significant"] = grp["n_significant"].shift(1)
df["prev_crucial"] = grp["n_crucial"].shift(1)
df["n_inspections_seen_so_far"] = grp.cumcount()

df["days_since_previous_inspection"] = (df["inspectionDate"] - df["prev_inspectionDate"]).dt.days
df["prev_non_pass"] = df["prev_status"].isin(["Conditional Pass", "Closed"]).astype(float)
df.loc[df["prev_status"].isna(), "prev_non_pass"] = np.nan

df[["estId", "inspectionDate", "source", "prev_inspectionDate", "days_since_previous_inspection",
    "prev_non_pass", "n_inspections_seen_so_far"]].head(10)

,estId,inspectionDate,source,prev_inspectionDate,days_since_previous_inspection,prev_non_pass,n_inspections_seen_so_far
0,001Vo000013PoSTIA0,2025-05-22,current,NaT,NaN,NaN,0
1,001Vo000013PoSTIA0,2025-10-02,current,2025-05-22,133.0,0.0,1
2,001Vo000013PoSTIA0,2026-02-03,current,2025-10-02,124.0,0.0,2
3,001Vo000013PoSTIA0,2026-07-14,current,2026-02-03,161.0,0.0,3
4,001Vo000013PoSTIA0,2026-07-21,current,2026-07-14,7.0,1.0,4
5,001Vo000013PoSUIA0,2025-05-08,current,NaT,NaN,NaN,0
6,001Vo000013PoSUIA0,2025-11-24,current,2025-05-08,200.0,0.0,1
7,001Vo000013PoSUIA0,2026-03-09,current,2025-11-24,105.0,0.0,2
8,001Vo000013PoSVIA0,2025-05-28,current,NaT,NaN,NaN,0
9,001Vo000013PoSVIA0,2026-06-04,current,2025-05-28,372.0,0.0,1


## 365-day recent history features

Same vectorized approach as before: for each row, find the earliest prior inspection within 365 days using `np.searchsorted` on the sorted date array, then take a cumulative-sum difference. Only rows with an earlier index (strictly earlier date) are ever included.

In [3]:
df["is_non_pass"] = df["inspectionStatus"].isin(["Conditional Pass", "Closed"]).astype(int)

def rolling_365(g):
    g = g.reset_index()
    dates = g["inspectionDate"].values.astype("datetime64[D]")
    total_infr = g["n_infractions"].to_numpy()
    crucial = g["n_crucial"].to_numpy()
    non_pass = g["is_non_pass"].to_numpy()

    cs_infr = np.concatenate([[0], np.cumsum(total_infr)])
    cs_crucial = np.concatenate([[0], np.cumsum(crucial)])
    cs_nonpass = np.concatenate([[0], np.cumsum(non_pass)])

    lower_edge = dates - np.timedelta64(365, "D")
    low_idx = np.searchsorted(dates, lower_edge, side="left")

    idx_arr = np.arange(len(g))
    return pd.DataFrame({
        "orig_index": g["index"],
        "prior_365d_n_inspections": idx_arr - low_idx,
        "prior_365d_n_non_pass": cs_nonpass[idx_arr] - cs_nonpass[low_idx],
        "prior_365d_total_infractions": cs_infr[idx_arr] - cs_infr[low_idx],
        "prior_365d_crucial_infractions": cs_crucial[idx_arr] - cs_crucial[low_idx],
    })

rolled = df.groupby("estId", group_keys=False).apply(rolling_365).set_index("orig_index")
df = df.join(rolled)
df[["estId", "inspectionDate", "prior_365d_n_inspections", "prior_365d_total_infractions"]].head(10)

,estId,inspectionDate,prior_365d_n_inspections,prior_365d_total_infractions
0,001Vo000013PoSTIA0,2025-05-22,0,0
1,001Vo000013PoSTIA0,2025-10-02,1,0
2,001Vo000013PoSTIA0,2026-02-03,2,2
3,001Vo000013PoSTIA0,2026-07-14,2,3
4,001Vo000013PoSTIA0,2026-07-21,3,5
5,001Vo000013PoSUIA0,2025-05-08,0,0
6,001Vo000013PoSUIA0,2025-11-24,1,0
7,001Vo000013PoSUIA0,2026-03-09,2,0
8,001Vo000013PoSVIA0,2025-05-28,0,0
9,001Vo000013PoSVIA0,2026-06-04,0,0


## Selecting the modern-period target rows

A target inspection must:

- have `inspectionDate >= 2023-11-10`
- come from the current source (not a historical row that happens to fall after the cutoff)
- have at least one previous inspection (historical or current)
- not be the Temporarily Not Operating inspection

In [4]:
candidates = df[df["inspectionDate"] >= CUTOFF].copy()
print("rows on or after the cutoff:", len(candidates))
print(candidates["source"].value_counts())

rows on or after the cutoff: 77629
source
current       77475
historical      154
Name: count, dtype: int64


There are a small number of historical-sourced rows dated after the cutoff. These are historical records for inspections the current export doesn't have on the same date, and they're dropped here since targets must come from the current source.

In [5]:
candidates = candidates[candidates["source"] == "current"].copy()

n_before_prev_filter = len(candidates)
candidates = candidates[candidates["n_inspections_seen_so_far"] > 0].copy()
n_no_prev = n_before_prev_filter - len(candidates)

n_before_tno_filter = len(candidates)
candidates = candidates[candidates["inspectionStatus"] != "Temporarily Not Operating"].copy()
n_tno_excluded = n_before_tno_filter - len(candidates)

candidates["target"] = candidates["inspectionStatus"].isin(["Conditional Pass", "Closed"]).astype(int)

print("rows lost, no previous inspection:", n_no_prev)
print("rows excluded, Temporarily Not Operating:", n_tno_excluded)
print("final modern target rows:", len(candidates))

rows lost, no previous inspection: 7709
rows excluded, Temporarily Not Operating: 0
final modern target rows: 69766


## Target distribution check

This should be close to the ~6% current-period Non-Pass rate found in the temporal distribution check.

In [6]:
print(candidates["target"].value_counts())
print("Non-Pass %:", round(100 * candidates["target"].mean(), 3))
print("earliest target date:", candidates["inspectionDate"].min())
print("latest target date:", candidates["inspectionDate"].max())

target
0    65543
1     4223
Name: count, dtype: int64
Non-Pass %: 6.053
earliest target date: 2023-11-10 00:00:00
latest target date: 2026-09-08 00:00:00


6.05% matches the current-period rate from the previous notebook, as expected since this is built the same way, just from scratch with the cutoff applied directly rather than filtered after the fact.

## Example establishment

In [7]:
example_est = candidates["estId"].value_counts().index[0]
cols = ["inspectionDate", "source", "inspectionStatus", "prev_inspectionDate", "days_since_previous_inspection",
        "prev_non_pass", "prior_365d_n_inspections", "target"]
candidates[candidates["estId"] == example_est][cols]

,inspectionDate,source,inspectionStatus,prev_inspectionDate,days_since_previous_inspection,prev_non_pass,prior_365d_n_inspections,target
5171,2023-12-13,current,Conditional Pass,2023-08-04,131.0,0.0,3,1
5172,2023-12-19,current,Conditional Pass,2023-12-13,6.0,1.0,4,1
5173,2023-12-27,current,Pass,2023-12-19,8.0,1.0,5,0
5174,2024-04-03,current,Conditional Pass,2023-12-27,98.0,0.0,5,1
5175,2024-04-05,current,Pass,2024-04-03,2.0,1.0,6,0
5176,2024-08-29,current,Conditional Pass,2024-04-05,146.0,0.0,5,1
5177,2024-09-06,current,Pass,2024-08-29,8.0,1.0,6,0
5178,2024-12-11,current,Pass,2024-09-06,96.0,0.0,7,0
5179,2025-03-18,current,Conditional Pass,2024-12-11,97.0,0.0,5,1
5180,2025-04-03,current,Conditional Pass,2025-03-18,16.0,1.0,6,1


## History availability

How many modern target rows actually have historical (pre-2023-11-10) inspection history behind them, versus only current-period history.

In [8]:
pre_cutoff_est = set(df.loc[df["inspectionDate"] < CUTOFF, "estId"].unique())
candidates["has_pre_cutoff_history"] = candidates["estId"].isin(pre_cutoff_est)

print("target rows with pre-2023-11-10 history:", candidates["has_pre_cutoff_history"].sum())
print("target rows without pre-2023-11-10 history:", (~candidates["has_pre_cutoff_history"]).sum())

target rows with pre-2023-11-10 history: 54107
target rows without pre-2023-11-10 history: 15659


## Feature summary

In [9]:
FEATURE_COLUMNS = [
    "days_since_previous_inspection",
    "prev_non_pass",
    "prev_total_infractions",
    "prev_minor",
    "prev_significant",
    "prev_crucial",
    "n_inspections_seen_so_far",
    "prior_365d_n_inspections",
    "prior_365d_n_non_pass",
    "prior_365d_total_infractions",
    "prior_365d_crucial_infractions",
]

identifier_cols = {"estId", "oldEstId", "_id", "unique_id"}
assert identifier_cols.isdisjoint(FEATURE_COLUMNS)

candidates[FEATURE_COLUMNS].describe()

,days_since_previous_inspection,prev_non_pass,prev_total_infractions,prev_minor,prev_significant,prev_crucial,n_inspections_seen_so_far,prior_365d_n_inspections,prior_365d_n_non_pass,prior_365d_total_infractions,prior_365d_crucial_infractions
count,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000,69766.000000
mean,196.657971,0.066264,0.986111,0.564946,0.311785,0.045624,17.075710,1.771049,0.171631,2.217398,0.141473
std,167.671439,0.248745,1.519861,0.896838,0.761675,0.274995,18.162161,1.161767,0.460726,3.451192,0.551012
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,109.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,1.000000,0.000000,0.000000,0.000000
50%,153.000000,0.000000,0.000000,0.000000,0.000000,0.000000,10.000000,2.000000,0.000000,1.000000,0.000000
75%,239.000000,0.000000,1.000000,1.000000,0.000000,0.000000,25.000000,2.000000,0.000000,3.000000,0.000000
max,3162.000000,1.000000,18.000000,8.000000,10.000000,5.000000,128.000000,9.000000,6.000000,50.000000,10.000000


## Leakage checks

In [10]:
assert (candidates["source"] == "current").all()
assert (candidates["inspectionDate"] >= CUTOFF).all()
assert (candidates["inspectionDate"] > candidates["prev_inspectionDate"]).all()
assert (candidates["prior_365d_n_inspections"] >= 0).all()
assert (candidates["inspectionStatus"] != "Temporarily Not Operating").all()
assert candidates.duplicated(subset=["estId", "inspectionDate"]).sum() == 0

for _, g in candidates.groupby("estId"):
    assert g["inspectionDate"].is_monotonic_increasing

print("all leakage checks passed")

all leakage checks passed


In [11]:
# brute-force check of the 365-day window on a random sample, same approach as the earlier notebook
rng = np.random.default_rng(0)
sample_idx = rng.choice(candidates.index, size=300, replace=False)

mismatches = 0
for i in sample_idx:
    row = candidates.loc[i]
    hist = df[(df["estId"] == row["estId"]) & (df["inspectionDate"] < row["inspectionDate"])]
    window = hist[hist["inspectionDate"] >= row["inspectionDate"] - pd.Timedelta(days=365)]
    if len(window) != row["prior_365d_n_inspections"]:
        mismatches += 1
    if window["n_infractions"].sum() != row["prior_365d_total_infractions"]:
        mismatches += 1

print("brute-force mismatches on 300 sampled rows (should be 0):", mismatches)

brute-force mismatches on 300 sampled rows (should be 0): 0


## Save output

In [12]:
output_cols = ["estId", "oldEstId", "inspectionDate", "inspectionStatus", "target",
               "prev_inspectionDate", "prev_status", "source", "has_pre_cutoff_history"] + FEATURE_COLUMNS

out = candidates[output_cols].sort_values(["estId", "inspectionDate"]).reset_index(drop=True)
out.to_csv("../outputs/prediction_events_current_v1.csv", index=False)
print("saved outputs/prediction_events_current_v1.csv with", len(out), "rows")

saved outputs/prediction_events_current_v1.csv with 69766 rows


## Summary

In [13]:
print("modern target prediction rows:", len(out))
print("establishments represented:", out["estId"].nunique())
print("earliest target date:", out["inspectionDate"].min())
print("latest target date:", out["inspectionDate"].max())
print("Pass count:", (out["target"] == 0).sum())
print("Non-Pass count:", (out["target"] == 1).sum())
print("Non-Pass %:", round(100 * out["target"].mean(), 3))
print("rows lost, no previous inspection:", n_no_prev)
print("rows excluded, Temporarily Not Operating:", n_tno_excluded)
print("target rows with pre-2023-11-10 history:", out["has_pre_cutoff_history"].sum())
print("target rows without pre-2023-11-10 history:", (~out["has_pre_cutoff_history"]).sum())
print("feature columns:", FEATURE_COLUMNS)

modern target prediction rows: 69766
establishments represented: 16779
earliest target date: 2023-11-10 00:00:00
latest target date: 2026-09-08 00:00:00
Pass count: 65543
Non-Pass count: 4223
Non-Pass %: 6.053
rows lost, no previous inspection: 7709
rows excluded, Temporarily Not Operating: 0
target rows with pre-2023-11-10 history: 54107
target rows without pre-2023-11-10 history: 15659
feature columns: ['days_since_previous_inspection', 'prev_non_pass', 'prev_total_infractions', 'prev_minor', 'prev_significant', 'prev_crucial', 'n_inspections_seen_so_far', 'prior_365d_n_inspections', 'prior_365d_n_non_pass', 'prior_365d_total_infractions', 'prior_365d_crucial_infractions']
